## Install the **libraries** needed for this notebook.

*`transformers`* provides access to pretrained Hugging Face models.

*`torch`* is the deep learning backend used to run the model.

*`pandas`* are used for data handling.

*`gdown`* downloads the dataset from Google Drive.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/Webinar3-Getting-Started-with-AI-Coding-and-Notebooks/blob/main/Notebook_with_HFmodel.ipynb)

In [1]:
# Import libraries for data analysis, model loading, and progress tracking.
# The pipeline function gives us a simple way to use pretrained Hugging Face models.

import pandas as pd
import numpy as np
import gdown

from transformers import pipeline
from tqdm.auto import tqdm

In [2]:
file_id = "1J424k-xkNvq349TyrD9m2s_iDywrHeds"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "workshop_survey_syn_dataset.csv",
    quiet=False
)

data = pd.read_csv(r'workshop_survey_syn_dataset.csv')

Downloading...
From: https://drive.google.com/uc?id=1J424k-xkNvq349TyrD9m2s_iDywrHeds
To: /content/workshop_survey_syn_dataset.csv
100%|██████████| 440k/440k [00:00<00:00, 69.4MB/s]


In [3]:
# Preview the first few rows of the dataset.
# This confirms that the dataset loaded correctly before we use the model.

data.head()

,participant_id,role,department,comfort_with_ai,main_interest_category,main_interest_detail,concern_category,concern_detail,free_text_feedback
0,837645,Faculty,Public Health,Comfortable,Grant Writing,Improving clarity and turnaround on competitiv...,Hallucinations and Accuracy,Hard to verify AI output accuracy for technica...,"I already use AI fairly regularly, I'd especia..."
1,475133,Undergraduate Student,Education,Neutral,Student Learning Support,Helping Education students use AI tutors witho...,Academic Misconduct,Students using AI to complete Education assign...,"I have mixed feelings about AI, I really want ..."
2,601099,Faculty,Biology,Uncomfortable,Teaching and Course Design,Designing AI-aware assignments and rubrics for...,Copyright and Intellectual Property,Unclear copyright and ownership rules for AI o...,"I've only dabbled with AI and feel uncertain, ..."
3,269917,Research Scientist,Computer Science,Comfortable,Data Analysis,"Cleaning, exploring, and visualizing Computer ...",Data Privacy,Unclear where AI vendors store and reuse our C...,"I already use AI fairly regularly, I'm keen to..."
4,698457,Administrative Staff,Medicine,Neutral,Research Productivity,Cutting time spent on repetitive analysis step...,Data Privacy,Protecting confidential participant and studen...,"I have mixed feelings about AI, I'd love pract..."


In [ ]:
# Define the support pathway labels that the model can choose from.
# In zero-shot classification, we provide the candidate labels instead of training the model on our own labeled examples.

support_pathways = [
    "AI Technical Consulting",
    "Teaching AI Literacy",
    "Architecture Development",
    "Infrastructure Support",
    "Research workflow consultation",
    "Teaching and Course Design Support",
    "Grant Writing Support",
    "Hackathons or workshop support",
    "Responsible AI establishment"
]

### Choosing a Model for Your Task

**The key principle:** Different models are designed for different tasks. You pick a model based on what you're trying to do.

For our task where we need the model to match participant feedback to support categories and the model should be good at understanding text and assigning labels. That's called **zero-shot classification**: the model learns which label fits the text best, without us training it.

We're using `facebook/bart-large-mnli` because it was built for exactly this kind of task—reading text and matching it to categories.

**That's it.** Don't worry about the details of how it works inside. Focus on: *What am I trying to do? Pick a model designed for that task.*

In [5]:
# Load a pretrained zero-shot classification model from Hugging Face.
# This model has already been trained, so we are using it for inference rather than training a new model.

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
sample = data.loc[12, "free_text_feedback"]

result = classifier(sample, candidate_labels=support_pathways)

result['labels'][0]

'Teaching and Course Design Support'

In [ ]:
# Wrap the Hugging Face model call inside a reusable function.
# The function returns the top recommended pathway and the model's confidence score.

def support_pathway_classifier(text):
  result = classifier(text, candidate_labels=support_pathways)
  top_label = result['labels'][0]
  top_score = result['scores'][0]

  return pd.Series({"recommended_pathway": top_label, "confidence_score": top_score})

In [ ]:

# Run the Hugging Face model on a small sample of rows.
# For a live webinar, starting with a small sample is faster and easier to inspect.

sample = data.head(4).copy()

sample_results = sample["free_text_feedback"].apply(support_pathway_classifier)

sample = pd.concat([sample, sample_results], axis=1)

sample[["participant_id","role","department","free_text_feedback","recommended_pathway","confidence_score"]]

,participant_id,role,department,free_text_feedback,recommended_pathway,confidence_score
0,837645,Faculty,Public Health,"I already use AI fairly regularly, I'd especia...",Grant Writing Support,0.831510
1,475133,Undergraduate Student,Education,"I have mixed feelings about AI, I really want ...",Responsible AI establishment,0.272447
2,601099,Faculty,Biology,"I've only dabbled with AI and feel uncertain, ...",Teaching and Course Design Support,0.369574
3,269917,Research Scientist,Computer Science,"I already use AI fairly regularly, I'm keen to...",Responsible AI establishment,0.366481
